# Bài 07 · Momentum, Adam & AdamW: đua ba bộ tối ưu từ thung lũng hẹp đến mạng MNIST

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/07-bo-toi-uu.ipynb)

Notebook đi kèm [Bài 07 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/) (demo tương tác nằm ở trang bài học).
Gradient descent ở Bài 05 chỉ có một nút vặn là tốc độ học $\eta$. Ở đây ta tự viết bằng numpy ba bộ tối ưu
**SGD, Momentum và Adam**, cho chúng đua trên thung lũng hẹp của demo rồi trên mạng MNIST 784 → 128 → 64 → 10
của khóa học, và xem warmup, lịch cosin và AdamW thay đổi điều gì.

**Bạn sẽ làm:**
1. Thấy vì sao SGD zig-zag trong một thung lũng hẹp, và vì sao $\eta$ bị chặn bởi hướng dốc nhất.
2. Viết Momentum và Adam (có hiệu chỉnh độ lệch), đua với SGD, rồi nghiêng thung lũng đi 20°.
3. Đua ba bộ trên mạng MNIST thật, tái tạo đúng đường cong của demo, rồi đọc kết quả quét $\eta$ × 3 seed.
4. Dùng warmup + cosin để huấn luyện ổn định với $\eta = 1$, mức mà $\eta$ cố định làm nổ.
5. So sánh Adam + phạt L2 với AdamW trên những trọng số không bao giờ nhận gradient.

**Cần biết trước:** [Bài 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gradient-descent/)
(tốc độ học, mini-batch) và [Bài 06 · Lan truyền ngược](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/)
(gradient của MLP, dùng lại ở mục 6). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo. Riêng kết quả quét đầy đủ
(84 lần huấn luyện 5 epoch) được nạp từ đúng tệp mà demo dùng, vì chạy lại mất vài phút; notebook chạy lại một phần
của nó và kiểm tra rằng ra cùng con số.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import gzip
import hashlib
import itertools
import json
import math
import urllib.request
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# Mỗi bộ tối ưu một màu, một dạng điểm và một kiểu nét, như trong demo
STYLE = {
    "SGD": {"color": BLUE, "marker": "o", "ls": "-"},
    "Momentum": {"color": ORANGE, "marker": "s", "ls": (0, (6, 3))},
    "Adam": {"color": GREEN, "marker": "^", "ls": (0, (1.2, 2.2))},
}


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = f"{x:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_eta(e):
    """η trong các bảng có một chữ số có nghĩa: 0,3 · 0,03 · 0,001 · 1."""
    return vn(e, max(0, -math.floor(math.log10(e) + 1e-9)))


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 0,001 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình sang kiểu Việt Nam. Trục log lấy vạch 1–2–5, hoặc lũy thừa của 10 nếu trục rất dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() == "log":
                lo, hi = sorted(lim)
                mult = (1, 2, 5) if hi / lo < 300 else (1,)
                axis.set_major_locator(
                    FixedLocator([m * 10.0**k for k in range(-6, 4) for m in mult if lo <= m * 10.0**k <= hi])
                )
                axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(vn_tick))


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.993527), vn_eta(0.003), vn_tick(0.05), vn_tick(-2.0))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28.
Ô dưới tải 4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu.

Giống demo, ta chia ba phần, mỗi phần một việc:
- **huấn luyện:** 55.000 ảnh đầu, để tính gradient;
- **kiểm định:** 5.000 ảnh cuối của tệp huấn luyện, để *chọn* $\eta$;
- **kiểm tra:** 10.000 ảnh của tệp kiểm tra, chỉ để báo kết quả cuối, không dùng để chọn gì.

Ô cũng nạp tệp kết quả quét mà demo dùng (`fetch_json`, có kiểm tra MD5).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của một lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
X_all, y_all = train_img.reshape(-1, 784) / 255.0, train_lab.astype(np.int64)
Xtr, ytr = X_all[:55000], y_all[:55000]  # huấn luyện
Xval, yval = X_all[55000:], y_all[55000:]  # kiểm định: chọn η ở đây
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(np.int64)  # kiểm tra: chỉ để báo cáo
runs = fetch_json("mnist-optimizers.json", "c4835a5fec46888af1bd11bcab00a947")  # kết quả quét của demo
print(f"huấn luyện {vn(len(ytr), 0)} · kiểm định {vn(len(yval), 0)} · kiểm tra {vn(len(yte), 0)} ảnh")
n_cfg, n_seed = len(runs["sweep"]), len(runs["seeds"])
print(f"tệp quét: {n_cfg} cấu hình (bộ tối ưu × η × lịch) × {n_seed} seed = {n_cfg * n_seed} lần chạy 5 epoch")
assert (len(ytr), len(yval), len(yte)) == (runs["nTrain"], runs["nVal"], runs["nTest"])

Mỗi chữ số một ảnh, lấy từ tập kiểm định:

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for d, ax in enumerate(axes):
    ax.imshow(Xval[np.flatnonzero(yval == d)[0]].reshape(28, 28), cmap="gray_r")
    ax.set_title(str(d), fontsize=11)
    ax.axis("off")
vn_axes(fig)
plt.show()

## 3 · SGD trong một thung lũng hẹp

Trước mạng thật, ta dùng thung lũng của demo, một hàm hai biến mà ta biết hết mọi thứ về nó:

$$L(\theta) = \tfrac12\left(0{,}04\,\theta_1^2 + 4\,\theta_2^2\right) = 0{,}02\,\theta_1^2 + 2\,\theta_2^2$$

Gradient là $g = (0{,}04\,\theta_1;\ 4\,\theta_2)$: theo $\theta_2$ dốc gấp 100 lần theo $\theta_1$. Hai số 0,04 và 4 là
**độ cong** theo hai hướng (trị riêng của ma trận Hessian). Mất mát của mạng nơ-ron cũng có những hướng rất dốc và
những hướng rất thoải như vậy, chỉ là trong hàng trăm nghìn chiều.

Mỗi bộ tối ưu là một hàm `step(theta, g, s, eta, t)`: nhận gradient `g` ở bước `t`, sửa `theta` tại chỗ, và giữ
trí nhớ của nó trong từ điển `s`. SGD (Bài 05) không cần trí nhớ: $\theta \leftarrow \theta - \eta\, g$.

Ô dưới cũng định nghĩa `race()`: chạy một bộ tối ưu 200 bước từ điểm $(-5{,}5;\ 2)$, như demo. “Về đích” nghĩa là
$L < 0{,}01$ và ở lại đó đến hết; “nổ” là khi $|\theta|$ vượt 1.000.

In [ ]:
LAM = np.array([0.04, 4.0])  # độ cong dọc đáy thung lũng và ngang thung lũng
TILT = math.radians(20)  # góc nghiêng dùng ở mục 5
START = np.array([-5.5, 2.0])
GOAL, N_STEPS, BOOM = 0.01, 200, 1e3


def rot(p, a):
    c, s = math.cos(a), math.sin(a)
    return np.array([c * p[0] - s * p[1], s * p[0] + c * p[1]])


def valley_loss(theta, tilt=0.0):
    u = rot(theta, -tilt)  # tọa độ trong hệ trục riêng của thung lũng
    return 0.5 * float((LAM * u**2).sum())


def valley_grad(theta, tilt=0.0):
    return rot(LAM * rot(theta, -tilt), tilt)


def sgd(theta, g, s, eta, t):
    theta -= eta * g


def race(step, eta, start=START, tilt=0.0, n=N_STEPS):
    """Chạy n bước. Trả về đường đi, mất mát, bước nổ (hoặc None) và bước về đích (hoặc None)."""
    theta, s = np.array(start, float), {}
    path, loss = [theta.copy()], [valley_loss(theta, tilt)]
    for t in range(1, n + 1):
        step(theta, valley_grad(theta, tilt), s, eta, t)
        path.append(theta.copy())
        loss.append(valley_loss(theta, tilt))
        if np.hypot(*theta) > BOOM:
            return np.array(path), np.array(loss), t, None
    bad = np.flatnonzero(np.array(loss) >= GOAL)  # về đích: từ bước này đến hết đều dưới GOAL
    arrive = None if len(bad) and bad[-1] == n else (int(bad[-1]) + 1 if len(bad) else 0)
    return np.array(path), np.array(loss), None, arrive


def outcome(r):
    return f"nổ ở bước {r[2]}" if r[2] else (f"về đích ở bước {r[3]}" if r[3] is not None else "chưa về đích")


r_sgd = race(sgd, 0.45)
d = r_sgd[0][1] - START
print(f"SGD, η = 0,45: {outcome(r_sgd)}; bước đầu Δθ = ({vn(d[0], 3)}; {vn(d[1], 3)})")
assert r_sgd[3] == 113, "khác với demo: hãy kiểm tra lại thung lũng"

Bước đầu tiên gần như chỉ đi theo $\theta_2$: gradient theo hướng dốc lớn hơn hẳn, nên SGD lao sang bờ bên kia.
Hình dưới vẽ đường đi trên các đường đồng mức (trái; dấu + là đáy, elip nét chấm là vùng đích), và hai tọa độ theo
từng bước (phải): $\theta_2$ (xám, vuông) đổi dấu mỗi bước, còn $\theta_1$ (đen, tròn) bò chậm về 0.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-6.5, 2.5, 361), np.linspace(-4.0, 2.5, 261))


def draw_valley(ax, tilt=0.0, title=""):
    u1, u2 = rot(np.stack([g1, g2]), -tilt)  # rot() cũng chạy trên cả lưới
    Lg = 0.5 * (LAM[0] * u1**2 + LAM[1] * u2**2)
    ax.contour(g1, g2, Lg, levels=[0.05, 0.2, 0.5, 1, 2, 4, 8], colors=GREY, linewidths=0.7)
    ax.contour(g1, g2, Lg, levels=[GOAL], colors="black", linewidths=1, linestyles=":")
    ax.plot(0, 0, "+", color="black", ms=10, mew=1.5)
    s0 = rot(START, tilt)
    ax.plot(s0[0], s0[1], "o", mfc="none", mec="black", ms=10, mew=1.5)
    ax.set(xlim=(-6.5, 2.5), ylim=(-4.0, 2.5), xlabel="θ₁", ylabel="θ₂", title=title)
    ax.set_aspect("equal")


def draw_path(ax, path, name, n=60, every=1):
    st = STYLE[name]
    p = path[: n + 1]
    ax.plot(
        p[:, 0],
        p[:, 1],
        color=st["color"],
        ls=st["ls"],
        lw=1.4,
        marker=st["marker"],
        ms=3.5,
        markevery=every,
        label=name,
    )


fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.3), gridspec_kw={"width_ratios": [1.5, 1]})
draw_valley(a, title="SGD, η = 0,45: 60 bước đầu")
draw_path(a, r_sgd[0], "SGD")
k = np.arange(61)
b.plot(k, r_sgd[0][:61, 0], color="black", marker="o", ms=3, lw=1.2, label="θ₁ (dọc đáy)")
b.plot(k, r_sgd[0][:61, 1], color=GREY, marker="s", ms=3, lw=1.2, ls=(0, (6, 3)), label="θ₂ (ngang thung lũng)")
b.axhline(0, color=GREY, lw=0.8)
b.set(xlabel="bước", ylabel="tọa độ", title="Hai tọa độ theo từng bước")
b.legend(frameon=False, fontsize=9)
vn_axes(fig)
plt.tight_layout()
plt.show()

Vì sao như vậy? Theo $\theta_2$, một bước SGD là $\theta_2 \leftarrow \theta_2 - \eta \cdot 4\theta_2 = (1 - 4\eta)\,\theta_2$;
theo $\theta_1$ là $\theta_1 \leftarrow (1 - 0{,}04\,\eta)\,\theta_1$. Mỗi hướng được nhân với một hệ số cố định mỗi bước:

- $1 - 4\eta < 0$ khi $\eta > 0{,}25$: $\theta_2$ đổi dấu mỗi bước, tức là nảy qua lại giữa hai bờ.
- $|1 - 4\eta| > 1$ khi $\eta > 0{,}5$: mỗi lần nảy xa hơn lần trước, SGD nổ.
- Theo $\theta_1$, hệ số $1 - 0{,}04\,\eta$ rất gần 1, nên dọc đáy SGD bò rất chậm.

Hướng dốc nhất đặt giới hạn cho $\eta$, hướng thoải nhất quyết định tốc độ. Ô dưới thử ba giá trị $\eta$:

In [ ]:
for eta in [0.25, 0.45, 0.55]:
    r = race(sgd, eta)
    print(f"η = {vn(eta)}: θ₂ ×{vn(1 - 4 * eta, 2):>6} mỗi bước, θ₁ ×{vn(1 - 0.04 * eta, 3)} mỗi bước → {outcome(r)}")
print(f"số điều kiện κ = 4 / {vn(LAM[0])} = {vn(LAM[1] / LAM[0], 0)}")

Với $\eta = 0{,}25$, SGD không nảy nhưng 200 bước vẫn chưa đủ để bò về đích; $\eta = 0{,}55$ thì nổ. Không có $\eta$
nào vừa không nảy vừa nhanh. Tỉ số giữa độ cong lớn nhất và nhỏ nhất (ở đây 4 / 0,04 = 100) gọi là **số điều kiện**
$\kappa$; $\kappa$ càng lớn, SGD càng chậm. Hai bộ tối ưu tiếp theo sửa vấn đề này theo hai cách khác nhau.

## 4 · Momentum: hòn bi có quán tính

Momentum (Polyak, 1964) giữ một **vận tốc** $v$, cộng dồn các gradient đã gặp:

$$v \leftarrow \beta\, v + g, \qquad \theta \leftarrow \theta - \eta\, v$$

Đây là dạng của PyTorch và của demo. (Chữ $v$ ở đây là vận tốc; ở Adam, mục 5, $v$ là một đại lượng khác.)
Trải công thức ra: $v_t = g_t + \beta g_{t-1} + \beta^2 g_{t-2} + \dots$. Gradient càng cũ càng nhẹ, theo lũy thừa
của $\beta$. Ở hướng mà gradient đổi dấu liên tục (ngang thung lũng), các số hạng triệt tiêu nhau. Ở hướng mà gradient
giữ nguyên dấu (dọc đáy), chúng cộng dồn, tối đa gấp $1/(1-\beta)$ lần:

In [ ]:
beta = 0.9
print("trọng số của g_t, g_{t−1}, g_{t−2}, …:", " · ".join(vn(beta**k, 3) for k in range(6)), "…")
print(f"tổng mọi trọng số = 1/(1 − β) = {vn(1 / (1 - beta), 0)}")


def momentum(theta, g, s, eta, t, beta=0.9):
    s["v"] = beta * s.get("v", 0.0) + g  # vận tốc: gradient mới + β × vận tốc cũ
    theta -= eta * s["v"]


r_mom = race(momentum, 0.45)
print(f"Momentum, η = 0,45, β = 0,9: {outcome(r_mom)} (SGD: {outcome(r_sgd)})")
print(f"θ₁ lớn nhất trên đường đi: {vn(r_mom[0][:, 0].max(), 2)} (xuất phát từ {vn(START[0], 1)}, đáy ở 0)")
assert r_mom[3] == 69, "khác với demo: hãy kiểm tra lại thung lũng"

Momentum về đích ở bước 69, sớm hơn SGD 44 bước, nhưng không đi thẳng: quán tính đưa nó vượt qua đáy tới
$\theta_1 = 1{,}50$ rồi mới quay lại.

### 🤔 Dự đoán trước
$\beta$ càng gần 1 thì quán tính càng lớn. Với cùng $\eta = 0{,}45$, Momentum với $\beta = 0{,}99$ về đích sớm hơn
hay muộn hơn $\beta = 0{,}9$?

In [ ]:
betas = [0, 0.5, 0.75, 0.9, 0.95, 0.99]
for b_ in betas:
    print(f"β = {vn(b_):>4}: {outcome(race(partial(momentum, beta=b_), 0.45))}")
beta_star = (1 - math.sqrt(0.45 * LAM[0])) ** 2
print(f"hãm vừa đủ theo hướng thoải: β* = (1 − √(η·0,04))² = {vn(beta_star, 3)}")
assert race(partial(momentum, beta=0), 0.45)[3] == r_sgd[3]  # β = 0 chính là SGD

Muộn hơn nhiều: $\beta = 0{,}99$ còn chưa về đích sau 200 bước. Quán tính lớn quá thì hòn bi lăn qua lăn lại quanh đáy
rất lâu mới dừng. $\beta = 0$ trùng khít SGD, và nhanh nhất là $\beta$ quanh 0,75. Đó là $\beta^*$ “hãm vừa đủ” theo
hướng thoải: nhỏ hơn thì còn bò, lớn hơn thì dao động (Goh, *Why Momentum Really Works*, 2017). $\beta = 0{,}9$
quen dùng trong thực tế không phải là tối ưu cho mọi bài toán.

Momentum còn chịu được $\eta$ lớn hơn. Theo hướng dốc, nó ổn định khi $\eta \cdot 4 < 2 + 2\beta$ (Goh, 2017), tức
$\eta < 0{,}95$ với $\beta = 0{,}9$, trong khi SGD cần $\eta < 0{,}5$:

In [ ]:
r1, r2 = race(sgd, 0.6), race(momentum, 0.6)
print(f"η = 0,6: SGD {outcome(r1)}, Momentum {outcome(r2)}; ngưỡng của Momentum (2 + 2β)/4 = {vn((2 + 2 * 0.9) / 4)}")
assert r1[2] == 19 and r2[3] == 69

## 5 · Adam: mỗi tham số một cỡ bước

Momentum vẫn dùng chung một $\eta$ cho mọi hướng. Adam (Kingma & Ba, 2015) chia bước của từng tham số cho độ lớn
gradient của *chính tham số đó*. Nó giữ hai trung bình trượt, $m$ của $g$ và $v$ của $g^2$, rồi bước theo tỉ số:

$$m \leftarrow \beta_1 m + (1-\beta_1)\, g \qquad v \leftarrow \beta_2 v + (1-\beta_2)\, g^2 \qquad
\theta \leftarrow \theta - \eta\, \frac{\hat m}{\sqrt{\hat v} + \epsilon}$$

với $\hat m = m/(1-\beta_1^t)$, $\hat v = v/(1-\beta_2^t)$, mặc định $\beta_1 = 0{,}9$, $\beta_2 = 0{,}999$,
$\epsilon = 10^{-8}$ (Algorithm 1 của bài báo). Mọi phép tính đều theo từng phần tử.

**Hiệu chỉnh độ lệch.** $m$ và $v$ bắt đầu từ 0, nên ở những bước đầu chúng bị kéo về 0: sau bước 1, $m = 0{,}1\,g$
và $v = 0{,}001\,g^2$. Chia cho $1 - \beta^t$ sửa đúng phần đó. Nếu bỏ đi, bước đầu dài $0{,}1/\sqrt{0{,}001} \approx 3{,}16$
lần $\eta$.

In [ ]:
def adam(theta, g, s, eta, t, b1=0.9, b2=0.999, eps=1e-8, correct=True):
    s["m"] = b1 * s.get("m", 0.0) + (1 - b1) * g  # trung bình trượt của g
    s["v"] = b2 * s.get("v", 0.0) + (1 - b2) * g * g  # trung bình trượt của g²
    m_hat = s["m"] / (1 - b1**t) if correct else s["m"]  # hiệu chỉnh độ lệch
    v_hat = s["v"] / (1 - b2**t) if correct else s["v"]
    theta -= eta * m_hat / (np.sqrt(v_hat) + eps)


g0 = valley_grad(START)
for correct in [True, False]:
    r = race(partial(adam, correct=correct), 0.2)
    d = r[0][1] - START
    print(
        f"Adam η = 0,2, {'có' if correct else 'không'} hiệu chỉnh: bước đầu Δθ = ({vn(d[0], 3)}; {vn(d[1], 3)}), {outcome(r)}"
    )
print(
    f"trong khi gradient ở điểm xuất phát là g = ({vn(g0[0], 2)}; {vn(g0[1], 0)}): lệch nhau {vn(g0[1] / -g0[0], 0)} lần"
)
print(f"không hiệu chỉnh, bước 1: m/√v = 0,1/√0,001 = {vn(0.1 / math.sqrt(0.001), 2)} lần η")
r_adam = race(adam, 0.2)
assert r_adam[3] == 61 and np.allclose(np.abs(r_adam[0][1] - START), 0.2, atol=1e-7)

Bước đầu của Adam dài đúng $\eta = 0{,}2$ theo **cả hai** trục, dù gradient theo $\theta_2$ lớn gấp 36 lần theo
$\theta_1$. Với hiệu chỉnh, $\hat m_1 / \sqrt{\hat v_1} = g/|g| = \pm 1$: bước đầu của Adam chỉ dùng *dấu* của gradient.
Từ đó Adam không còn bị hướng dốc chặn $\eta$: nó về đích ở bước 61, sớm hơn cả Momentum.

Adam không phải lúc nào cũng bước dài $\eta$. $\sqrt{\hat v}$ nhớ khoảng 1.000 bước gradient ($\beta_2 = 0{,}999$),
$\hat m$ chỉ nhớ khoảng 10 bước. Khi gradient đổi dấu (gần đáy), $\hat m$ nhỏ đi trước; khi gradient nhỏ dần,
$\sqrt{\hat v}$ còn nhớ những gradient lớn cũ. Cả hai đều làm bước ngắn lại. Ô dưới đo độ dài bước trên thung lũng,
và trên hàm một biến $L = \theta^2/2$, nơi gradient không bao giờ đổi dấu:

In [ ]:
step_len = np.linalg.norm(np.diff(r_adam[0], axis=0), axis=1)
print("thung lũng, |Δθ| ở bước 1 · 10 · 50 · 100:", " · ".join(vn(step_len[k - 1], 4) for k in [1, 10, 50, 100]))
theta1, s1, steps1 = np.array([5.0]), {}, []
for t in range(1, 401):
    before = theta1.copy()
    adam(theta1, theta1.copy(), s1, 0.01, t)  # L = θ²/2 nên g = θ
    steps1.append(float(before[0] - theta1[0]))
print(
    f"L = θ²/2 từ θ = 5, η = 0,01: bước 1 dài {vn(steps1[0], 4)}, bước 400 dài {vn(steps1[-1], 4)} "
    f"({pct(steps1[-1] / 0.01, 0)} của η); θ vẫn dương: {bool(theta1[0] > 0)}"
)
print(f"trí nhớ: β₁ = 0,9 → ~{vn(1 / (1 - 0.9), 0)} bước, β₂ = 0,999 → ~{vn(1 / (1 - 0.999), 0)} bước")

Trên thung lũng, bước đầu dài 0,2828 = 0,2·√2 (đường chéo của một bước ±0,2 theo mỗi trục), và gần đáy chỉ còn
0,0041: gradient đổi dấu, nên $\hat m$ triệt tiêu. Trên $L = \theta^2/2$, gradient không đổi dấu lần nào, vậy mà sau
400 bước bước chỉ còn 57% của $\eta$. Lý do là $\sqrt{\hat v}$ vẫn nhớ những gradient lớn lúc đầu, trong khi
$\hat m$ đã theo kịp gradient nhỏ hiện tại. Vậy bước của Adam ngắn lại khi gradient nhiễu, khi nó đổi dấu, và cả khi
nó nhỏ dần đều đặn.

### 🤔 Dự đoán trước
Nghiêng thung lũng đi 20° (cùng hình dạng, cùng điểm xuất phát so với đáy, chỉ xoay hệ trục). Bộ nào trong ba bộ đổi
kết quả, và đổi theo hướng nào?

In [ ]:
results = {}
for name, step, eta in [("SGD", sgd, 0.45), ("Momentum", momentum, 0.45), ("Adam", adam, 0.2)]:
    flat_r, tilted_r = race(step, eta), race(step, eta, start=rot(START, TILT), tilt=TILT)
    results[name] = (flat_r, tilted_r)
    print(f"{name:>8}: dọc theo trục {outcome(flat_r):>20}  ·  nghiêng 20° {outcome(tilted_r)}")
assert [results[n][1][3] for n in results] == [113, 69, 168], "khác với demo: hãy kiểm tra lại thung lũng"
d = np.diff(results["Adam"][1][0][:4], axis=0)
print(
    f"nghiêng 20°: 3 bước đầu của Adam có hướng {' · '.join(vn(a, 1) for a in np.degrees(np.arctan2(d[:, 1], d[:, 0])))} độ"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, j, title in [(axes[0], 0, "Dọc theo trục"), (axes[1], 1, "Nghiêng 20°")]:
    draw_valley(ax, tilt=TILT * j, title=title)
    for name in results:
        draw_path(ax, results[name][j][0], name, n=N_STEPS, every=4)
axes[0].legend(frameon=False, fontsize=9, loc="lower right")
vn_axes(fig)
plt.tight_layout()
plt.show()

SGD và Momentum cho kết quả y hệt: công thức của chúng chỉ dùng vectơ $g$, nên xoay hệ trục thì đường đi xoay theo.
Adam thì chậm hẳn, từ bước 61 thành bước 168. Nó chia gradient cho độ lớn riêng của trục $\theta_1$ và trục $\theta_2$,
nên chỉ có lợi khi hướng dốc và hướng thoải nằm dọc theo các trục, tức là dọc theo từng tham số. Khi đáy thung lũng
nằm chéo, mỗi tham số đều thấy cả phần dốc lẫn phần thoải. Trong hình phải, những bước đầu của Adam (xanh lá, tam
giác) đi theo đường chéo −45° của hệ trục, vì mỗi tọa độ chỉ bước theo dấu gradient của nó, chứ không theo hình dạng
của thung lũng.

Mạng thật nằm giữa hai trường hợp này. Vì vậy câu hỏi “Adam có thắng không” phải trả lời bằng thí nghiệm.

## 6 · Đua trên mạng MNIST thật

Mạng của khóa học (Bài 04–06): $h_1 = \mathrm{ReLU}(xW_1 + b_1)$, $h_2 = \mathrm{ReLU}(h_1W_2 + b_2)$,
$z = h_2W_3 + b_3$, softmax + cross-entropy, khởi tạo He, batch 64. Lan truyền ngược là của Bài 06. Hàm `train()` dưới
đây làm đúng như `reel/mnist_mlp/optimizers.py`, script tạo dữ liệu cho demo:
- cùng seed thì cùng trọng số đầu và cùng thứ tự batch, nên khác biệt chỉ còn ở bộ tối ưu;
- mỗi lượt (epoch) là 860 bước, và mất mát kiểm định được đo sau mỗi ¼ epoch (215 bước);
- một lần chạy “nổ” khi mất mát của một batch không còn là số hữu hạn.

In [ ]:
SIZES, BATCH, STEPS = [784, 128, 64, 10], 64, 4300  # 5 epoch × 860 bước, như demo


def forward(params, x):
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = np.maximum(0, x @ W1 + b1)
    h2 = np.maximum(0, h1 @ W2 + b2)
    return h1, h2, h2 @ W3 + b3


def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def evaluate(params, X, y):
    z = forward(params, X)[2]
    p = softmax(z)
    return float(-np.log(p[np.arange(len(y)), y]).mean()), float((z.argmax(1) == y).mean())


def backward(params, x, y):
    """Mất mát của batch và gradient theo thứ tự W1, b1, W2, b2, W3, b3 (Bài 06)."""
    h1, h2, z = forward(params, x)
    p = softmax(z)
    loss = -np.log(p[np.arange(len(y)), y]).mean()
    dz = p.copy()
    dz[np.arange(len(y)), y] -= 1
    dz /= len(y)
    _, (W2, _), (W3, _) = params
    dh2 = (dz @ W3.T) * (h2 > 0)
    dh1 = (dh2 @ W2.T) * (h1 > 0)
    return float(loss), [x.T @ dh1, dh1.sum(0), h1.T @ dh2, dh2.sum(0), h2.T @ dz, dz.sum(0)]


def lr_at(eta, schedule, t, warmup=0.05, floor=0.1):
    """η ở bước t: cố định, hoặc warmup tuyến tính 5% số bước rồi giảm theo cosin xuống 10% η (mục 8)."""
    if schedule == "fixed":
        return eta
    w = round(warmup * STEPS)
    if t <= w:
        return eta * t / w
    return eta * (floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * (t - w) / (STEPS - w))))


def train(step, eta, schedule="fixed", seed=0, n=STEPS, step_bias=None):
    """n bước của bộ tối ưu `step` (bias dùng `step_bias` nếu có). Trả về tham số, nhật ký và trí nhớ của bộ tối ưu."""
    rng = np.random.default_rng(seed)
    params = [[rng.normal(0, np.sqrt(2 / a), (a, b)), np.zeros(b)] for a, b in itertools.pairwise(SIZES)]  # He
    flat = [a for layer in params for a in layer]
    states, rules = [{} for _ in flat], [step, step_bias or step] * 3
    log, t = {"loss": [], "val": [evaluate(params, Xval, yval)], "boom": None}, 0
    with np.errstate(all="ignore"):  # một lần chạy nổ sẽ tràn số; ta tự phát hiện bên dưới
        while t < n:
            order = rng.permutation(len(Xtr))
            for i in range(0, len(order), BATCH):
                idx = order[i : i + BATCH]
                loss, grads = backward(params, Xtr[idx], ytr[idx])
                t += 1
                log["loss"].append(loss)
                if not np.isfinite(loss):
                    log["boom"] = t
                    return params, log, states
                for a, g, s, rule in zip(flat, grads, states, rules):
                    rule(a, g, s, lr_at(eta, schedule, t), t)
                if t % 215 == 0:
                    log["val"].append(evaluate(params, Xval, yval))
                if t == n:
                    break
    return params, log, states


print(f"{vn(sum(a * b + b for a, b in itertools.pairwise(SIZES)), 0)} tham số; {len(Xtr) // BATCH + 1} bước mỗi epoch")

### 🤔 Dự đoán trước
Demo đã thử vài $\eta$ cho mỗi bộ tối ưu và giữ $\eta$ có mất mát kiểm định thấp nhất sau 5 epoch. Với Momentum
$\beta = 0{,}9$ (dạng $v \leftarrow \beta v + g$ của mục 4), $\eta$ tốt nhất lớn hơn hay nhỏ hơn của SGD, và
khoảng bao nhiêu lần?

In [ ]:
best = {(r["opt"], r["schedule"]): r for r in runs["sweep"] if r.get("best")}
NAMES = {"sgd": "SGD", "momentum": "Momentum", "adam": "Adam"}
for opt, name in NAMES.items():
    grid = [r["eta"] for r in runs["sweep"] if r["opt"] == opt and r["schedule"] == "fixed"]
    print(
        f"{name:>8}: thử η ∈ {{{'; '.join(vn_eta(e) for e in grid)}}} → tốt nhất η = {vn_eta(best[(opt, 'fixed')]['eta'])}"
    )
for sch in ["fixed", "cosine"]:
    ratio = best[("sgd", sch)]["eta"] / best[("momentum", sch)]["eta"]
    print(f"lịch {'cố định' if sch == 'fixed' else 'warmup + cosin'}: η của SGD / η của Momentum = {vn(ratio, 0)}")
print(f"1/(1 − β) = {vn(1 / (1 - runs['beta']), 0)}")

Nhỏ hơn 10 lần, ở cả hai lịch. Ở dạng $v \leftarrow \beta v + g$, theo một hướng mà gradient gần như không đổi,
$v$ tiến tới $g/(1-\beta) = 10\,g$, nên Momentum với $\eta$ đi gần giống SGD với $10\eta$. So sánh “cùng $\eta$” giữa
hai bộ là không công bằng. (Lưới $\eta$ thưa, mỗi nấc gấp khoảng 3 lần, nên đây là “khoảng 10”, không phải đúng 10.)
Adam có thang $\eta$ riêng: với nó, $\eta$ là độ dài bước, không nhân với gradient.

Giờ ta chạy lại 1 epoch đầu của ba lần chạy tốt nhất (seed 0), và so với đường cong mà demo lưu:

In [ ]:
OPTS = {"SGD": ("sgd", sgd), "Momentum": ("momentum", momentum), "Adam": ("adam", adam)}
live = {}
for name, (key, step) in OPTS.items():
    eta = best[(key, "fixed")]["eta"]
    live[name] = train(step, eta, n=860)
    log = live[name][1]
    ref = next(c for c in runs["curves"] if c["opt"] == key and c["schedule"] == "fixed")
    got = [v for v, _ in log["val"]]
    smooth = np.array(log["loss"]).reshape(-1, 43).mean(1)  # trung bình mỗi 43 bước, như demo
    print(
        f"{name:>8}, η = {vn_eta(eta)}: kiểm định sau 0 · ¼ · ½ · ¾ · 1 epoch: {' · '.join(vn(v, 4) for v in got)}"
        f"  →  đúng {pct(log['val'][-1][1])}"
    )
    assert np.allclose(got, ref["val"][:5], atol=5e-4) and np.allclose(smooth, ref["train"][:20], atol=5e-4), (
        "khác với demo: hãy kiểm tra lại dữ liệu"
    )
    assert abs(log["val"][-1][1] - ref["valAcc"][4]) < 0.001, "khác với demo: hãy kiểm tra lại dữ liệu"
print("trùng với đường cong seed 0 mà demo lưu (sai khác < 0,0005)")

Hình dưới vẽ mất mát huấn luyện (trung bình mỗi 43 bước, thang log) của 1 epoch đó. Mỗi bộ một màu, một dạng điểm
và một kiểu nét, như trong demo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
for name in OPTS:
    smooth = np.array(live[name][1]["loss"]).reshape(-1, 43).mean(1)
    st = STYLE[name]
    eta = best[(OPTS[name][0], "fixed")]["eta"]
    ax.plot(
        43 * np.arange(1, 21),
        smooth,
        color=st["color"],
        ls=st["ls"],
        marker=st["marker"],
        ms=4,
        lw=1.6,
        label=f"{name}, η = {vn_eta(eta)}",
    )
ax.set_yscale("log")
ax.set(
    xlabel="bước (1 epoch = 860 bước)",
    ylabel="mất mát huấn luyện",
    title="1 epoch đầu, seed 0, mỗi bộ với η tốt nhất của nó",
)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3, which="both")
vn_axes(fig)
plt.show()

Sau 1 epoch ba đường còn sát nhau và còn đang đi xuống. Một seed, một epoch thì chưa nói được bộ nào tốt hơn. Phần
còn lại của cuộc đua (5 epoch, 3 seed, cả lưới $\eta$, hai lịch) mất vài phút, nên ta đọc kết quả mà demo đã lưu.

## 7 · Kết quả đầy đủ: 3 seed, cả lưới $\eta$

Bảng dưới lấy, cho mỗi bộ và mỗi lịch, $\eta$ có mất mát kiểm định trung bình thấp nhất. Khoảng trong ngoặc là
min–max của 3 seed. Độ chính xác trên tập kiểm tra chỉ để báo cáo; nó không được dùng để chọn gì.

In [ ]:
def seed_range(r, key="valLoss", d=4):
    v = r[key]
    return f"{vn(np.mean(v), d)} ({vn(min(v), d)}–{vn(max(v), d)})"


for sch, label in [("fixed", "η cố định"), ("cosine", "warmup 5% + cosin")]:
    print(label)
    for opt, name in NAMES.items():
        r = best[(opt, sch)]
        print(
            f"  {name:>8}  η = {vn_eta(r['eta']):<5}  mất mát kiểm định {seed_range(r)}  "
            f"đúng kiểm định {pct(np.mean(r['valAcc']))}  kiểm tra {pct(np.mean(r['testAcc']))}"
        )
    rows = [best[(o, sch)] for o in NAMES]
    overlap = max(min(r["valLoss"]) for r in rows) <= min(max(r["valLoss"]) for r in rows)
    print(f"  khoảng 3 seed của cả ba chồng lên nhau: {overlap}")
gain = [max(best[(o, "cosine")]["valLoss"]) < min(best[(o, "fixed")]["valLoss"]) for o in NAMES]
stuck = next(r for r in runs["sweep"] if r["opt"] == "momentum" and r["schedule"] == "fixed" and r["eta"] == 0.3)
print(f"Momentum η = 0,3 cố định: đúng kiểm định {' · '.join(pct(a) for a in stuck['valAcc'])} (3 seed)")
print(f"warmup + cosin tốt hơn η cố định ở cả 3 seed, cho từng bộ: {gain}")

Với $\eta$ cố định, cả ba bộ về gần cùng một chỗ: khoảng 3 seed của chúng chồng lên nhau, tức là khác biệt nằm
**trong dao động giữa các seed**. Trên mạng nhỏ này, SGD được chỉnh $\eta$ kỹ không thua Adam. Điều này trùng với
Choi và cộng sự (2019): so sánh bộ tối ưu phụ thuộc rất nhiều vào việc chỉnh siêu tham số kỹ đến đâu.

Đổi sang warmup + cosin thì cả ba đều tốt hơn rõ, vượt ra ngoài dao động giữa các seed, còn giữa ba bộ với nhau thì vẫn
nằm trong dao động đó. Ở bài toán này, **lịch tốc độ học quan trọng hơn việc chọn bộ nào**.

Hình dưới vẽ cả lưới: mất mát kiểm định (trung bình 3 seed, vạch là min–max) theo $\eta$, thang log ở cả hai trục.
Điểm tô đặc là $\eta$ tốt nhất; điểm có chữ “nổ” ở trên cùng là $\eta$ làm ít nhất một seed nổ.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (sch, title) in zip(axes, [("fixed", "η cố định"), ("cosine", "Warmup 5% + cosin")]):
    for opt, name in NAMES.items():
        st = STYLE[name]
        rows = [r for r in runs["sweep"] if r["opt"] == opt and r["schedule"] == sch]
        ok = [r for r in rows if r["valLoss"]]
        e = np.array([r["eta"] for r in ok])
        m = np.array([np.mean(r["valLoss"]) for r in ok])
        err = [m - [min(r["valLoss"]) for r in ok], [max(r["valLoss"]) for r in ok] - m]
        ax.errorbar(
            e,
            m,
            yerr=err,
            color=st["color"],
            ls=st["ls"],
            marker=st["marker"],
            mfc="white",
            ms=6,
            lw=1.4,
            capsize=3,
            label=name,
        )
        b = best[(opt, sch)]
        ax.plot(b["eta"], np.mean(b["valLoss"]), st["marker"], color=st["color"], ms=8)
        for r in rows:
            if not r["valLoss"]:
                ax.plot(r["eta"], 2.6, st["marker"], color=st["color"], mfc="white", ms=6)
                ax.annotate(
                    f"nổ {r['diverged']}/3",
                    (r["eta"], 2.6),
                    xytext=(0, -14),
                    textcoords="offset points",
                    ha="center",
                    fontsize=8,
                )
    ax.set(xscale="log", yscale="log", ylim=(0.05, 3.2), xlabel="η (thang log)", title=title)
    ax.grid(alpha=0.3, which="both")
axes[0].set_ylabel("mất mát kiểm định sau 5 epoch")
axes[0].legend(frameon=False, fontsize=9, loc="upper left")
vn_axes(fig)
plt.tight_layout()
plt.show()

Mỗi bộ có một “thung lũng” $\eta$ riêng, và thang của chúng lệch nhau: SGD quanh 0,3–1, Momentum nhỏ hơn 10 lần,
Adam quanh 0,001–0,003. Hai phía của thung lũng đều dốc. $\eta$ nhỏ quá thì sau 5 epoch vẫn chưa học xong; lớn quá
thì nổ (SGD $\eta = 3$) hoặc kẹt ở mất mát cao (Momentum $\eta = 0{,}3$: đúng 9,66–30,04%, gần như đoán mò).

Đường cong theo epoch của ba lần chạy tốt nhất (seed 0, cũng từ tệp của demo; điểm cuối là sau 5 epoch):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), sharey=True)
for ax, (sch, title) in zip(axes, [("fixed", "η cố định"), ("cosine", "Warmup 5% + cosin")]):
    for opt, name in NAMES.items():
        c = next(c for c in runs["curves"] if c["opt"] == opt and c["schedule"] == sch)
        st = STYLE[name]
        ax.plot(
            np.arange(len(c["val"])) / 4,
            c["val"],
            color=st["color"],
            ls=st["ls"],
            marker=st["marker"],
            ms=3.5,
            lw=1.5,
            label=f"{name}, η = {vn_eta(c['eta'])}",
        )
    ax.set(yscale="log", ylim=(0.05, 3), xlabel="epoch", title=title)
    ax.grid(alpha=0.3, which="both")
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("mất mát kiểm định")
vn_axes(fig)
plt.tight_layout()
plt.show()
print(
    "mất mát kiểm định cuối (seed 0):",
    " · ".join(f"{c['opt']}/{c['schedule']} {vn(c['val'][-1], 4)}" for c in runs["curves"]),
)

## 8 · Warmup và lịch giảm tốc độ học

Lịch “warmup + cosin” của demo (cũng là dạng của LLaMA và nanoGPT):
- **warmup:** trong 5% số bước đầu (215 bước), $\eta$ tăng tuyến tính từ 0 lên $\eta$ đỉnh;
- **cosin:** sau đó $\eta$ giảm theo nửa chu kỳ cosin, xuống 10% $\eta$ đỉnh ở bước cuối.

Trong bảng ở mục 7, SGD với $\eta = 1$ cố định nổ ở 1/3 seed, còn có warmup thì $\eta = 1$ lại là $\eta$ tốt nhất.
Ô dưới chạy lại 215 bước đầu (đúng phần warmup, ¼ epoch) của cả hai lịch cho 3 seed. Trong mỗi seed, hai lịch xuất phát từ cùng trọng số và thấy
cùng các batch.

In [ ]:
w_steps = round(runs["warmup"] * STEPS)
print(
    f"warmup {pct(runs['warmup'], 0)} = {w_steps} bước; cosin xuống {pct(runs['floor'], 0)} η đỉnh ở bước {vn(STEPS, 0)}"
)
warm = {}
for seed in range(3):
    for sch in ["fixed", "cosine"]:
        _, log, _ = train(sgd, 1.0, sch, seed=seed, n=215)
        warm[(seed, sch)] = log
        L = np.array(log["loss"])
        end = f"nổ ở bước {log['boom']}" if log["boom"] else f"sau 215 bước: kiểm định {vn(log['val'][-1][0], 3)}"
        print(
            f"seed {seed}, {('η = 1 cố định' if sch == 'fixed' else 'η = 1 + warmup'):<14}: "
            f"mất mát batch lớn nhất {vn(L[np.isfinite(L)].max(), 1):>5} · {end}"
        )
row = next(r for r in runs["sweep"] if r["opt"] == "sgd" and r["schedule"] == "fixed" and r["eta"] == 1)
n_boom = sum(warm[(s, "fixed")]["boom"] is not None for s in range(3))
print(f"số seed nổ với η = 1 cố định: {n_boom}/3 (demo, sau 5 epoch: {row['diverged']}/3)")
assert warm[(2, "fixed")]["boom"] == 8 and n_boom == row["diverged"], "khác với demo: hãy kiểm tra lại dữ liệu"
assert all(warm[(s, "cosine")]["boom"] is None for s in range(3))

Hình trái vẽ hai lịch, $\eta(t)/\eta_{\text{đỉnh}}$ theo epoch. Hình phải vẽ mất mát từng batch của 100 bước đầu
(thang log): $\eta$ cố định là đường cam nét gạch, có warmup là đường xanh liền; dạng điểm phân biệt 3 seed, dấu ×
đen là chỗ nổ.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.3), gridspec_kw={"width_ratios": [1, 1.4]})
t_all = np.arange(1, STEPS + 1)
a.plot(t_all / 860, [lr_at(1, "fixed", t) for t in t_all], color=ORANGE, ls=(0, (6, 3)), lw=2, label="cố định")
a.plot(t_all / 860, [lr_at(1, "cosine", t) for t in t_all], color=BLUE, lw=2, label="warmup 5% + cosin")
a.set(xlabel="epoch", ylabel="η(t) / η đỉnh", title="Hai lịch tốc độ học", ylim=(0, 1.08))
a.legend(frameon=False, fontsize=9)
for seed, mk in zip(range(3), ["o", "s", "^"]):
    for sch, color, ls in [("fixed", ORANGE, (0, (6, 3))), ("cosine", BLUE, "-")]:
        L = np.array(warm[(seed, sch)]["loss"][:100])
        k = np.arange(1, len(L) + 1)
        ok = np.isfinite(L)
        b.plot(k[ok], L[ok], color=color, ls=ls, lw=1.2, marker=mk, ms=4, markevery=10, label=f"seed {seed}")
        if not ok.all():
            b.plot(k[ok][-1], L[ok][-1], "x", color="black", ms=10, mew=2)
            b.annotate("nổ", (k[ok][-1], L[ok][-1]), xytext=(6, 0), textcoords="offset points", va="center")
b.set(yscale="log", xlabel="bước", ylabel="mất mát của batch", title="SGD, η = 1: 100 bước đầu")
handles, labels = b.get_legend_handles_labels()
b.legend(handles[1::2], labels[1::2], frameon=False, fontsize=9, title="xanh: warmup · cam: cố định", title_fontsize=9)
vn_axes(fig)
plt.tight_layout()
plt.show()

Với $\eta = 1$ ngay từ bước đầu, mất mát của một batch vọt lên 8,3 / 54,8 / 32,3 ở 3 seed (lúc đầu khoảng 2,4), và
seed 2 nổ ở bước 8. Có warmup thì mất mát batch lớn nhất chỉ 2,3–2,4, ngang mức lúc đầu, và giảm dần từ đó.

Vì sao warmup giúp? Ở mục 3, SGD ổn định khi $\eta$ nhỏ hơn 2 chia cho độ cong lớn nhất. Kalra & Barkeshli (2024) đo
độ cong đó trong lúc huấn luyện và thấy rằng lợi ích chính của warmup là cho phép dùng $\eta$ đỉnh lớn hơn: những bước
nhỏ đầu tiên đưa mạng tới vùng có độ cong nhỏ hơn, nơi $\eta$ lớn trở nên ổn định. Notebook này không đo độ cong; ta
chỉ thấy kết quả của nó.

Lấy đúng seed vừa nổ, cũng $\eta = 1$, và cho warmup + cosin chạy hết 5 epoch. Đây là một ô trong bảng của demo, và ta
kiểm tra rằng chạy lại ra cùng con số:

In [ ]:
params_w, log_w, _ = train(sgd, 1.0, "cosine", seed=2)
val_loss, val_acc = log_w["val"][-1]
test_acc = evaluate(params_w, Xte, yte)[1]
row = next(r for r in runs["sweep"] if r["opt"] == "sgd" and r["schedule"] == "cosine" and r["eta"] == 1)
print(
    f"SGD, η = 1, warmup + cosin, seed 2, 5 epoch: kiểm định {vn(val_loss, 4)} (đúng {pct(val_acc)}), "
    f"kiểm tra đúng {pct(test_acc)}"
)
print(
    f"demo, cùng ô: kiểm định {vn(row['valLoss'][2], 4)} (đúng {pct(row['valAcc'][2])}), kiểm tra đúng {pct(row['testAcc'][2])}"
)
assert abs(val_loss - row["valLoss"][2]) < 5e-4 and abs(val_acc - row["valAcc"][2]) < 0.001, "khác với demo"
assert abs(test_acc - row["testAcc"][2]) < 0.001, "khác với demo: hãy kiểm tra lại dữ liệu"

Cùng trọng số đầu, cùng các batch, cùng $\eta$ đỉnh: một lịch nổ ở bước 8, lịch kia thuộc cấu hình tốt nhất của SGD.

Phần giảm dần ở cuối cũng có ích. Khi $\eta$ nhỏ lại, nhiễu của mini-batch nhỏ theo, nên các bước cuối tinh chỉnh quanh
đáy thay vì nảy quanh nó. Các mô hình ngôn ngữ lớn dùng đúng các lịch này:
- LLaMA: 2.000 bước warmup, rồi cosin xuống 10% $\eta$ đỉnh (Touvron et al., 2023, mục 2.3);
- Kimi K2: lịch WSD (warmup – giữ – giảm), 500 bước warmup, giữ $\eta = 2 \cdot 10^{-4}$ trong 10 nghìn tỉ token đầu,
  rồi cosin xuống $2 \cdot 10^{-5}$ (Kimi Team, 2025).

## 9 · AdamW: weight decay tách riêng

**Weight decay** co trọng số về 0 một chút sau mỗi bước, để mạng không dựa vào những trọng số quá lớn (Bài 08 sẽ nói
vì sao điều này giúp). Có hai cách viết:

- **Phạt L2:** cộng $\lambda\theta$ vào gradient, tức $g \leftarrow g + \lambda\theta$, rồi đưa vào bộ tối ưu.
- **Tách riêng (decoupled):** co trực tiếp $\theta \leftarrow \theta - \eta\lambda\theta$, ngoài bộ tối ưu.

Với SGD, hai cách là một: bước SGD trên $g + \lambda\theta$ chính là phép co cộng với bước SGD thường.
Loshchilov & Hutter (2019) chỉ ra rằng với Adam thì không. Phần $\lambda\theta$ cũng bị chia cho $\sqrt{\hat v}$, nên
trọng số có gradient lớn gần như không bị phạt. AdamW tách phép co ra khỏi Adam:

$$\theta \leftarrow \theta - \eta\left(\frac{\hat m}{\sqrt{\hat v} + \epsilon} + \lambda\theta\right)$$

(Đây là dạng của PyTorch, có nhân $\lambda$ với $\eta$; bài báo gốc nhân $\lambda$ với hệ số của lịch, không nhân với
$\eta$. Chữ $\lambda$ ở đây cũng khác $\lambda$ của phạt L2 ở Bài 08.)

Ta thử trên đúng mạng MNIST. Một số điểm ảnh ở viền **không bao giờ có nét** trong tập huấn luyện, nên hàng
tương ứng của $W_1$ không bao giờ nhận gradient từ dữ liệu. Không có weight decay, Adam để yên chúng: $m = v = 0$, nên
bước bằng $0/(0 + \epsilon) = 0$. Weight decay chỉ áp cho ma trận $W$, không cho độ lệch $b$, như thường làm.

In [ ]:
def adam_l2(theta, g, s, eta, t, lam=0.0):
    adam(theta, g + lam * theta, s, eta, t)  # phạt L2: λθ đi vào gradient, nên cũng bị Adam chia cho √v̂


def adamw(theta, g, s, eta, t, lam=0.0):
    theta -= eta * lam * theta  # weight decay tách riêng: co mọi trọng số cùng một tỉ lệ
    adam(theta, g, s, eta, t)


N_DECAY = 430  # ½ epoch
dead = (Xtr == 0).all(axis=0)  # điểm ảnh không có nét trong bất kỳ ảnh huấn luyện nào
W1_init = np.random.default_rng(0).normal(0, np.sqrt(2 / 784), (784, 128))  # đúng W1 đầu của train(seed=0)
params_base, _, states_base = train(adam, 1e-3, n=N_DECAY)  # Adam η = 0,001, seed 0, không weight decay
print(f"{dead.sum()} điểm ảnh không bao giờ có nét → {vn(dead.sum() * 128, 0)} trọng số của W1 không nhận gradient")
same = np.array_equal(params_base[0][0][dead], W1_init[dead])
print(f"không weight decay, sau {N_DECAY} bước Adam: các trọng số đó vẫn y như lúc đầu: {same}")
assert dead.sum() == 67 and np.array_equal(params_base[0][0][dead], W1_init[dead])

### 🤔 Dự đoán trước
Thêm phạt L2 vào Adam ($\eta = 0{,}001$, ½ epoch). Với $\lambda = 0{,}01$ và với $\lambda = 0{,}0001$ (nhỏ hơn 100 lần),
các trọng số “không có nét” này co lại bao nhiêu? Còn với AdamW $\lambda = 0{,}1$?

In [ ]:
decay = {
    "Adam + L2, λ = 0,01": partial(adam_l2, lam=1e-2),
    "Adam + L2, λ = 0,0001": partial(adam_l2, lam=1e-4),
    "AdamW, λ = 0,1": partial(adamw, lam=0.1),
}
dec_runs = {}
w0 = np.abs(W1_init[dead])
print(f"lúc đầu: |w| trung bình {vn(w0.mean(), 4)}, lớn nhất {vn(w0.max(), 4)}")
for label, rule in decay.items():
    params, log, _ = train(rule, 1e-3, n=N_DECAY, step_bias=adam)
    dec_runs[label] = params
    frac = np.abs(params[0][0][dead]) / w0  # mỗi trọng số còn bao nhiêu phần so với lúc đầu
    print(
        f"{label:>22}: |w| còn (trung vị) {pct(np.median(frac))}, (lớn nhất) {pct(frac.max())} so với lúc đầu"
        f" · kiểm định {vn(log['val'][-1][0], 4)}"
    )
print(f"AdamW: (1 − η·λ)^{N_DECAY} = (1 − 0,001·0,1)^{N_DECAY} = {vn((1 - 1e-3 * 0.1) ** N_DECAY, 4)}")
print(
    f"với L2, mỗi bước dài ≈ η: |w| trung vị {vn(np.median(w0), 3)} cần khoảng {vn(np.median(w0) / 1e-3, 0)} bước để về 0"
)
medians = [np.median(np.abs(dec_runs[k][0][0][dead]) / w0) for k in list(decay)[:2]]
assert max(medians) < 1e-6  # phạt L2: gần như về 0, với cả hai λ
assert np.allclose(dec_runs["AdamW, λ = 0,1"][0][0][dead], W1_init[dead] * (1 - 1e-4) ** N_DECAY)

Với phạt L2, cả hai $\lambda$ đều xóa gần sạch các trọng số đó: λ nhỏ đi 100 lần mà kết quả không đổi. Lý do là
gradient của chúng chỉ còn $\lambda\theta$, và Adam chia nó cho chính độ lớn của nó, nên mỗi bước dài khoảng $\eta$
**bất kể $\lambda$ lớn hay nhỏ**. Sau khoảng $|\theta|/\eta$ bước (vài chục bước) trọng số về tới 0. Với AdamW, mọi
trọng số này co đúng theo $(1 - \eta\lambda)$ mỗi bước, khớp với công thức đến từng chữ số.

Còn những trọng số *có* gradient thì sao? Hình dưới chia các trọng số còn lại của $W_1$ thành 8 nhóm bằng nhau theo độ
lớn gradient $\sqrt{\hat v}$ của lần chạy không weight decay (trái: nhỏ, phải: lớn). Mỗi điểm là trung vị của
$|w| / |w_{\text{không decay}}|$ trong nhóm: 1 là không co chút nào.

In [ ]:
active = np.broadcast_to(~dead[:, None], W1_init.shape)
rms = np.sqrt(states_base[0]["v"] / (1 - 0.999**N_DECAY))[active]  # độ lớn gradient của từng trọng số
groups = np.array_split(np.argsort(rms, kind="stable"), 8)
base_w = np.abs(params_base[0][0][active])
fig, ax = plt.subplots(figsize=(8, 3.4))
for (label, params), (color, mk, ls) in zip(
    dec_runs.items(), [(ORANGE, "s", (0, (6, 3))), (ORANGE, "D", ":"), (GREEN, "^", "-")]
):
    w = np.abs(params[0][0][active])
    med = [np.median(w[g] / base_w[g]) for g in groups]
    ax.plot(range(1, 9), med, color=color, marker=mk, ls=ls, lw=1.6, ms=7, label=label)
    print(f"{label:>22}: trung vị |w|/|w không decay| theo nhóm 1…8: {' · '.join(vn(m, 2) for m in med)}")
ax.axhline(1, color=GREY, lw=0.8)
ax.set(
    xlabel="nhóm trọng số theo độ lớn gradient (1 = nhỏ nhất, 8 = lớn nhất)",
    ylabel="|w| / |w không decay|",
    ylim=(-0.03, 1.08),
)
ax.set_title("Phạt L2 trong Adam co trọng số có gradient nhỏ nhiều hơn hẳn; AdamW co đều", fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="lower right")
vn_axes(fig)
plt.show()

Phạt L2 trong Adam co mạnh nhất những trọng số có gradient nhỏ, và càng ít với những trọng số có gradient lớn: với
$\lambda = 0{,}01$, từ 0,00 ở nhóm 1 đến 0,64 ở nhóm 8. Đó đúng là điều Loshchilov & Hutter mô tả. Mức co còn phụ
thuộc vào $\lambda$ theo kiểu khó đoán: $\lambda$ nhỏ đi 100 lần vẫn xóa sạch nhóm 1. AdamW co mọi nhóm gần như cùng
một tỉ lệ, 0,96–0,97, sát phép co thuần 0,9579 (ít hơn một chút, vì gradient của dữ liệu kéo bù lại một phần).

Hệ quả thực tế: với AdamW, $\lambda$ giữ đúng nghĩa “mỗi bước co bao nhiêu phần”, và $\lambda$ tốt nhất ít phụ thuộc
vào $\eta$ hơn (Loshchilov & Hutter, 2019). Vì vậy AdamW là lựa chọn mặc định để huấn luyện Transformer:
- LLaMA: AdamW với $\beta_1 = 0{,}9$, $\beta_2 = 0{,}95$, $\lambda = 0{,}1$, cắt gradient ở 1,0 (Touvron et al., 2023);
- mặc định của PyTorch: $\eta = 0{,}001$, $\beta_2 = 0{,}999$, $\lambda = 0{,}01$.

$\beta_2 = 0{,}95$ cho $\hat v$ trí nhớ ngắn hơn (khoảng 20 bước thay vì 1.000), nên Adam phản ứng nhanh hơn khi độ
lớn gradient đột ngột tăng.

## 10 · Bộ tối ưu mới: Muon

Adam chỉnh cỡ bước theo từng *phần tử*. Muon (Keller Jordan, 2024) nhìn cả *ma trận*: nó lấy vận tốc $V$ của Momentum
cho một ma trận trọng số và thay $V$ bằng ma trận gần nhất có mọi giá trị kỳ dị bằng 1. Nếu $V = USV^\top$ (SVD) thì
bước cập nhật là $UV^\top$, tức mọi hướng trong bước đều dài như nhau. Thay vì SVD, Muon dùng 5 lần lặp
Newton–Schulz chỉ gồm phép nhân ma trận. Ta áp nó lên vận tốc của $W_2$ (ma trận ẩn 128×64) sau 1 epoch Momentum ở
mục 6:

In [ ]:
def newton_schulz5(G, steps=5):
    """Đẩy mọi giá trị kỳ dị của G về gần 1, chỉ bằng phép nhân ma trận (hệ số của Keller Jordan, 2024)."""
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G / np.linalg.norm(G)
    tall = X.shape[0] > X.shape[1]
    X = X.T if tall else X
    for _ in range(steps):
        A = X @ X.T
        X = a * X + (b * A + c * A @ A) @ X
    return X.T if tall else X


V = live["Momentum"][2][2]["v"]  # vận tốc của W2 sau 860 bước
s_before = np.linalg.svd(V, compute_uv=False)
s_after = np.linalg.svd(newton_schulz5(V), compute_uv=False)
print(f"vận tốc của W2: {V.shape[0]}×{V.shape[1]}, {len(s_before)} giá trị kỳ dị")
print(
    f"trước: từ {vn(s_before.min(), 4)} đến {vn(s_before.max(), 3)} (lớn nhất / nhỏ nhất = {vn(s_before.max() / s_before.min(), 0)})"
)
print(
    f"sau 5 lần lặp: từ {vn(s_after.min(), 2)} đến {vn(s_after.max(), 2)}; trong [0,5; 1,3]: {((s_after > 0.5) & (s_after < 1.3)).sum()}/{len(s_after)}"
)

Trước, vài hướng chiếm gần hết bước cập nhật; sau 5 lần lặp, 63/64 hướng có độ dài trong khoảng 0,5–1,3 (không đúng
bằng 1: hệ số được chọn để hội tụ nhanh, chấp nhận sai lệch). Muon chỉ dùng cho các ma trận ẩn; embedding, lớp đầu ra, độ lệch và
các hệ số chuẩn hóa vẫn dùng AdamW. Kimi K2 (1 nghìn tỉ tham số) được huấn luyện bằng MuonClip, một biến thể của Muon
(Kimi Team, 2025). Notebook này không huấn luyện với Muon; xem bài viết của Keller Jordan nếu muốn thử.

## 11 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Hiệu chỉnh độ lệch của Adam, trên MNIST
Trên thung lũng, bỏ hiệu chỉnh chỉ làm bước đầu dài gấp khoảng 3,16 lần. Trên mạng thật, ta chạy 100 bước đầu với
một $\eta$ nhỏ và một $\eta$ lớn:

In [ ]:
for eta in [1e-3, 3e-2]:
    for correct in [True, False]:
        params_bc, log, _ = train(partial(adam, correct=correct), eta, n=100)
        print(
            f"Adam η = {vn_eta(eta)}, {'có' if correct else 'không'} hiệu chỉnh: mất mát batch lớn nhất "
            f"{vn(max(log['loss']), 1):>4}, kiểm định sau 100 bước {vn(evaluate(params_bc, Xval, yval)[0], 3)}"
        )

Với $\eta$ nhỏ, bỏ hiệu chỉnh còn có lợi một chút: những bước đầu dài hơn khoảng 3 lần, giống một $\eta$ lớn hơn.
Với $\eta = 0{,}03$, chính những bước dài đó làm mất mát của một batch vọt lên 58,2, và sau 100 bước mạng vẫn kém xa
(kiểm định 1,946 so với 0,337). Hiệu chỉnh độ lệch giữ cho bước đầu không dài quá $\eta$, dù $\eta$ là bao nhiêu.

### Góc nghiêng của thung lũng và $\eta$ của Adam
Ở mục 5, nghiêng 20° làm Adam chậm đi gần 3 lần. Đổi góc và $\eta$ để xem Adam phụ thuộc vào hệ trục đến mức nào
(SGD và Momentum thì không phụ thuộc chút nào):

In [ ]:
for deg in [0, 5, 20, 45]:
    a_ = math.radians(deg)
    line = " · ".join(f"η = {vn(e)}: {outcome(race(adam, e, start=rot(START, a_), tilt=a_))}" for e in [0.05, 0.2, 0.5])
    print(f"nghiêng {deg:>2}°  {line}")

### Điểm xuất phát
Đổi `start` để xuất phát gần đáy hơn hoặc xa hơn. Bộ nào được lợi nhiều nhất khi xuất phát ngay trên đáy thung lũng,
tức là $\theta_2 = 0$?

In [ ]:
for start in [(-5.5, 2.0), (-5.5, 0.0), (-2.0, 2.0)]:
    print(
        f"xuất phát ({vn(start[0], 1)}; {vn(start[1], 1)}): "
        + " · ".join(
            f"{n} {outcome(race(s, e, start=start))}"
            for n, (s, e) in {"SGD": (sgd, 0.45), "Momentum": (momentum, 0.45), "Adam": (adam, 0.2)}.items()
        )
    )

Xuất phát ngay trên đáy ($\theta_2 = 0$): SGD vẫn về đích ở bước 113, vì tốc độ của nó do hướng thoải quyết định,
còn Momentum và Adam nhanh gấp đôi so với lúc xuất phát ở $\theta_2 = 2$. Xuất phát gần đáy hơn theo $\theta_1$,
tại $(-2;\ 2)$, thì SGD lại về đích sớm nhất. Không bộ nào thắng ở mọi điểm xuất phát.

## 12 · Giới hạn: tối ưu tốt hơn chưa chắc đoán tốt hơn

Bộ tối ưu chỉ làm một việc: giảm mất mát trên **dữ liệu huấn luyện**. Ô dưới so mất mát huấn luyện (trung bình 43
batch cuối) với mất mát kiểm định cuối của 6 đường cong seed 0 mà demo lưu.

In [ ]:
for c in runs["curves"]:
    print(
        f"{NAMES[c['opt']]:>8} · {'cố định' if c['schedule'] == 'fixed' else 'warmup + cosin':<14}: "
        f"huấn luyện {vn(c['train'][-1], 4)} · kiểm định {vn(c['val'][-1], 4)} · kiểm định / huấn luyện = "
        f"{vn(c['val'][-1] / c['train'][-1], 1)}"
    )

Adam với warmup + cosin đẩy mất mát huấn luyện xuống thấp nhất, 0,0148, thấp hơn khoảng 3 lần so với SGD cố định
(0,0461). Nhưng mất mát kiểm định chỉ giảm từ 0,0772 xuống 0,0649. Tỉ số kiểm định / huấn luyện tăng từ 1,5–1,7
(η cố định) lên 2,5–4,4 (warmup + cosin): mạng bắt đầu học thuộc những ảnh nó đã thấy. Tối ưu tốt hơn trên tập
huấn luyện không tự động thành đoán tốt hơn trên ảnh mới.
Có những bài toán mà Adam tổng quát hóa kém hơn SGD (Wilson et al., 2017), dù khi được chỉnh siêu tham số đủ kỹ thì
Adam không thua (Choi et al., 2019).

Weight decay ở mục 9 là một cách chống học thuộc. Bài 08 sẽ đo quá khớp một cách có hệ thống, và thêm phạt L2,
dropout và tăng cường dữ liệu.

## Tóm tắt
- Trong thung lũng có số điều kiện $\kappa = 100$, SGD bị hướng dốc chặn $\eta$ (nảy khi $\eta > 0{,}25$, nổ khi
  $\eta > 0{,}5$) và bò chậm theo hướng thoải: về đích ở bước 113.
- Momentum $v \leftarrow \beta v + g$ cộng dồn gradient, bước càng cũ càng nhẹ; về đích ở bước 69, chịu được $\eta$ tới
  $(2 + 2\beta)/4$. $\beta$ lớn nhất không phải tốt nhất: $\beta = 0{,}99$ chưa về đích sau 200 bước.
- Adam chia cho $\sqrt{\hat v}$ theo từng tham số; hiệu chỉnh độ lệch làm bước đầu dài đúng $\eta$ theo mỗi trục. Nó về
  đích ở bước 61, nhưng nghiêng thung lũng 20° thì thành bước 168: lợi thế của nó gắn với hệ trục.
- Trên MNIST, η tốt nhất của Momentum nhỏ hơn SGD 10 lần ($1/(1-\beta)$). Với $\eta$ cố định, ba bộ ngang nhau trong
  dao động giữa các seed; warmup + cosin giúp cả ba vượt ra ngoài dao động đó.
- Warmup cho phép $\eta = 1$: cố định thì seed 2 nổ ở bước 8; có warmup thì cùng seed đó thuộc cấu hình tốt nhất của SGD.
- Phạt L2 trong Adam bị chia cho $\sqrt{\hat v}$, nên co trọng số không đều và không theo $\lambda$; AdamW co mọi trọng số
  theo $(1 - \eta\lambda)$ mỗi bước.

## Câu hỏi ôn tập
1. Trong thung lũng $L = 0{,}02\,\theta_1^2 + 2\,\theta_2^2$, vì sao không có $\eta$ nào cho SGD vừa không nảy vừa nhanh?
2. Vì sao $\eta$ tốt nhất của Momentum dạng $v \leftarrow \beta v + g$ nhỏ hơn của SGD khoảng 10 lần khi $\beta = 0{,}9$?
3. Bước đầu tiên của Adam có hiệu chỉnh độ lệch dài bao nhiêu theo mỗi trục? Bỏ hiệu chỉnh thì sao?
4. Xoay thung lũng không đổi gì với SGD và Momentum nhưng làm Adam chậm đi. Vì sao?
5. Với $\eta = 1$, SGD cố định nổ ở seed 2 còn có warmup thì không, dù cùng trọng số đầu và cùng các batch. Warmup thay
   đổi điều gì ở những bước đầu?
6. Một trọng số không bao giờ nhận gradient từ dữ liệu. Với Adam + phạt L2, vì sao nó về 0 nhanh như nhau dù
   $\lambda = 0{,}01$ hay $0{,}0001$? AdamW làm gì với nó?

<details><summary>Gợi ý đáp án</summary>

1. Theo hướng dốc, mỗi bước nhân $\theta_2$ với $1 - 4\eta$: không nảy cần $\eta \le 0{,}25$, không nổ cần $\eta < 0{,}5$.
   Theo hướng thoải, mỗi bước chỉ nhân $\theta_1$ với $1 - 0{,}04\,\eta \ge 0{,}98$, nên với mọi $\eta$ được phép, SGD bò chậm.
2. Theo một hướng mà gradient gần như không đổi, $v$ tiến tới $g/(1-\beta) = 10\,g$, nên Momentum với $\eta$ đi gần
   giống SGD với $10\eta$.
3. Đúng $\eta$: $\hat m_1/\sqrt{\hat v_1} = g/|g| = \pm 1$. Bỏ hiệu chỉnh thì $m_1/\sqrt{v_1} = 0{,}1/\sqrt{0{,}001}$, tức
   khoảng 3,16 lần $\eta$; trên MNIST với $\eta = 0{,}03$, điều đó làm mất mát vọt lên.
4. SGD và Momentum chỉ dùng vectơ $g$, nên đường đi xoay theo hệ trục. Adam chia từng thành phần cho độ lớn riêng của
   nó, nên chỉ có lợi khi hướng dốc và hướng thoải nằm dọc theo các tham số.
5. Các bước đầu dùng $\eta$ nhỏ, nên không có bước nào đủ lớn để làm mất mát vọt lên; theo Kalra & Barkeshli (2024),
   chúng đưa mạng tới vùng có độ cong nhỏ hơn, nơi $\eta$ lớn trở nên ổn định.
6. Gradient của nó chỉ là $\lambda\theta$, và Adam chia nó cho chính độ lớn của nó, nên mỗi bước dài khoảng $\eta$
   bất kể $\lambda$. AdamW nhân nó với $(1 - \eta\lambda)$ mỗi bước, đúng như $\lambda$ quy định.
</details>

## Tìm hiểu thêm
- B. T. Polyak (1964), [Some methods of speeding up the convergence of iteration methods](https://doi.org/10.1016/0041-5553(64)90137-5): momentum “hòn bi nặng”.
- G. Goh (2017), [Why Momentum Really Works](https://distill.pub/2017/momentum/), Distill: momentum trên hàm bậc hai,
  có hình tương tác; nguồn của ngưỡng $2 + 2\beta$ và $\beta^*$.
- D. Kingma & J. Ba (2015), [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980): Algorithm 1
  và mục 3 về hiệu chỉnh độ lệch.
- I. Loshchilov & F. Hutter (2019), [Decoupled Weight Decay Regularization](https://arxiv.org/abs/1711.05101): AdamW.
- D. Choi et al. (2019), [On Empirical Comparisons of Optimizers for Deep Learning](https://arxiv.org/abs/1910.05446):
  xếp hạng bộ tối ưu phụ thuộc vào cách chỉnh siêu tham số.
- A. Wilson et al. (2017), [The Marginal Value of Adaptive Gradient Methods in Machine Learning](https://arxiv.org/abs/1705.08292).
- D. S. Kalra & M. Barkeshli (2024), [Why Warmup the Learning Rate?](https://arxiv.org/abs/2406.09405)
- H. Touvron et al. (2023), [LLaMA](https://arxiv.org/abs/2302.13971), mục 2.3: AdamW và lịch cosin của một mô hình
  ngôn ngữ lớn.
- K. Jordan (2024), [Muon: An optimizer for hidden layers in neural networks](https://kellerjordan.github.io/posts/muon/).
- Kimi Team (2025), [Kimi K2: Open Agentic Intelligence](https://arxiv.org/abs/2507.20534): MuonClip và lịch WSD.

**Bài tiếp theo:** [Bài 08 · Quá khớp & chính quy hóa](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/chong-qua-khop/):
khi mạng học thuộc dữ liệu huấn luyện, và cách chống lại bằng phạt L2, dropout và tăng cường dữ liệu.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")